## Import Data and Libraries

In [ ]:
# Uncomment the following lines if you are running this notebook in Colab. You'd then need to put the dataset in your drive directory
# from google.colab import drive
# drive.mount('/content/drive')

import csv

import math
import matplotlib.pyplot as plt
import numpy as np

import os
import sys
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.utils import save_image

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

%pip install sigmf
%pip install scikit-commpy

import sigmf
from sigmf import SigMFFile, sigmffile
import datetime as dt
import random
import warnings
import pickle

from scipy.signal import convolve
from commpy.modulation import PSKModem, QAMModem
from commpy.filters import rrcosfilter, rcosfilter

DATASET_PATH = f"/content/drive/MyDrive/path/to/dataset"

%cd $DATASET_PATH

# Custom Functions

## MITRF Helper Functions

In [ ]:
# QSPK Constants

# Helper Lambda Function
get_sinr = lambda s, i: 10*np.log10(np.mean(np.abs(s)**2)/np.mean(np.abs(i)**2))

# Parameters for QPSK
mod_num = 4
mod = PSKModem(mod_num)

rolloff = 0.5
Fs = 25e6
oversample_factor = 16
Ts = oversample_factor/Fs

tVec, sPSF = rrcosfilter(oversample_factor*8, rolloff, Ts, Fs)
tVec, sPSF = tVec[1:], sPSF[1:]
sPSF = sPSF.astype(np.complex64)

seg_len = int(2**15 + 2**13)
n_sym = seg_len//oversample_factor

window_len = 40960
get_rand_start_idx = lambda sig_len: np.random.randint(sig_len-window_len)
num_train_frame = {"EMISignal1": 530, "CommSignal2": 100, "CommSignal3": 139}
num_trainval_frame = {"EMISignal1": 580, "CommSignal2": 150, "CommSignal3": 189}

# (optional) cache interference frames count
n_train_frame_types = {'EMISignal1':530, 'CommSignal2':100, 'CommSignal3':139}

# Inverse of the Gray-map used earlier: (0,0)->0, (0,1)->1, (1,1)->2, (1,0)->3
_class_to_pair = {
    0: (0, 0),
    1: (0, 1),
    2: (1, 1),
    3: (1, 0),
}

# --------- bits -> 4-class symbol labels (Gray map used earlier) ----------
_pair_to_class = {(0,0):0, (0,1):1, (1,1):2, (1,0):3}

N = None # used samples (in case we want to use_subset)
C = None # number of channels
Lwin = None # signal window size


In [ ]:
# ========================= MIT RF Challenge Utilities pulled: https://github.com/RFChallenge/rfchallenge_singlechannel_starter ===== #
def generate_qpsk_signal(n_sym=n_sym, mod=mod, oversample_factor=oversample_factor, sPSF=sPSF, Fc=0, Fs=Fs):
    sB = np.random.randint(2, size=n_sym*mod.num_bits_symbol)  # Random bit stream
    sQ = mod.modulate(sB)  # Modulated baud points
    sQ_padded = np.zeros(len(sQ)*oversample_factor, dtype=np.complex64)
    start_idx = oversample_factor//2
    sQ_padded[start_idx::oversample_factor] = sQ

    sig = convolve(sQ_padded, sPSF, 'same') # Waveform with PSF
    sig *= np.exp(2*np.pi*1j*np.arange(len(sig))*Fc/Fs, dtype=np.complex64)
    return sig, sQ_padded, sQ, sB

def matched_filter(sig, sPSF, Fc=0, Fs=Fs):
    sig_filt = sig*np.exp(-2*np.pi*1j*np.arange(len(sig))*Fc/Fs, dtype=np.complex64)
    sig_filt = convolve(sig_filt, sPSF/np.sum(sPSF), 'same')
    return sig_filt

def read_sigmf_file(filename=f'sample', folderpath=None):

    # Load a dataset
    full_filename = filename
    if folderpath is not None:
        full_filename = os.path.join(folderpath, filename)

    meta = sigmffile.fromfile(full_filename)

    # Get some metadata and all annotations
    sample_rate = meta.get_global_field(SigMFFile.SAMPLE_RATE_KEY)
    sample_count = meta.sample_count
    signal_duration = sample_count / sample_rate

    data = meta.read_samples(0, meta.sample_count)
    return data, meta

def load_dataset_sample(idx, dataset_type, sig_type):
    foldername = os.path.join('dataset',dataset_type,sig_type)
    filename = f'{sig_type}_{dataset_type}_{idx:04d}'
    # Special handling for "Separation" validation and test set; only using Comm2 vs [sig_type] for this iteration
    if 'sep_' in dataset_type:
        filename = f'CommSignal2_vs_{sig_type}_{dataset_type}_{idx:04d}'
    data, meta = read_sigmf_file(filename=filename, folderpath=foldername)
    return data, meta

def fit_minmax_scaler(X_train, eps=1e-6):
    x_min = X_train.min(axis=0, keepdims=True)
    x_max = X_train.max(axis=0, keepdims=True)
    x_range = np.clip(x_max - x_min, eps, None)
    return x_min.astype(np.float32), x_max.astype(np.float32), x_range.astype(np.float32)

def apply_minmax_scaler(X, x_min, x_range, multiplier = 5, clip=True):
    X_scaled = 2.0 * ((X - x_min) / x_range) - 1.0
    if clip:
        X_scaled = np.clip(X_scaled, -1.0, 1.0)
    return (multiplier * X_scaled).astype(np.float32)

def apply_local_minmax(X):
    """
    X shape: (N, 2*window_len)
    Scales each sample individually to [-1, 1] while preserving 0 at the origin.
    """
    # Find the maximum absolute value for each sample across all features
    X_maxabs = np.max(np.abs(X), axis=1, keepdims=True)

    # Divide by max absolute value, adding 1e-9 to prevent div-by-zero
    X_scaled = X / (X_maxabs + 1e-9)
    return X_scaled

# ========================= MIT RF Challenge Helpers ===== #
def bits_to_symbol_classes(bits):
    b = np.asarray(bits, dtype=np.int64).reshape(-1)
    pairs = b.reshape(-1, 2)  # (2560,2)
    y = np.array([_pair_to_class[tuple(p)] for p in pairs], dtype=np.int64)  # (2560,)
    return y

def slice_symbol_windows_X(X, sps=16, tau0=8, window_len=127, n_sym=2560):
    assert X.ndim == 2 and X.shape[0] == 2
    N = X.shape[1]
    half_left = window_len // 2
    half_right = window_len - half_left - 1

    centers = tau0 + sps * np.arange(n_sym, dtype=np.int64)
    W = np.zeros((n_sym, 2, window_len), dtype=np.float32)

    for k, c in enumerate(centers):
        left = int(c) - half_left
        right = int(c) + half_right  # inclusive
        src_l = max(left, 0)
        src_r = min(right, N-1)
        dst_l = src_l - left
        dst_r = dst_l + (src_r - src_l)
        W[k, :, dst_l:dst_r+1] = X[:, src_l:src_r+1]

    return W, centers

# --------- RRC pulse + matched filter ----------
def rrc_pulse(beta=0.5, sps=16, span_symbols=8):
    N = span_symbols * sps
    t = np.arange(-N/2, N/2 + 1) / sps
    g = np.zeros_like(t, dtype=np.float64)

    for i, ti in enumerate(t):
        if abs(ti) < 1e-12:
            g[i] = (1.0 + beta*(4/np.pi - 1))
        elif beta > 0 and abs(abs(ti) - 1/(4*beta)) < 1e-12:
            g[i] = (beta/np.sqrt(2)) * ((1 + 2/np.pi)*np.sin(np.pi/(4*beta)) +
                                        (1 - 2/np.pi)*np.cos(np.pi/(4*beta)))
        else:
            num = np.sin(np.pi*ti*(1-beta)) + 4*beta*ti*np.cos(np.pi*ti*(1+beta))
            den = np.pi*ti*(1 - (4*beta*ti)**2)
            g[i] = num / den

    g = g / (np.linalg.norm(g) + 1e-12)
    return g.astype(np.float32)

# --------- robust extractor for generate_qpsk_signal() ----------
def matched_filter_full(sig_complex):
    # Convolve on full-rate samples (before decimation)
    return np.convolve(sig_complex.astype(np.complex64), h_mf, mode="same").astype(np.complex64)

def generate_soi_and_bits():
    out = generate_qpsk_signal()
    if not isinstance(out, (list, tuple)):
        raise RuntimeError("generate_qpsk_signal() returned a non-tuple; cannot parse.")
    sig1 = None
    bits = None

    if sq_padded == True:
      for item in out:
          if isinstance(item, np.ndarray):
              if item.ndim == 1 and item.size == sig_len and np.iscomplexobj(item):
                  sig1 = item.astype(np.complex64)
              if item.ndim == 1 and item.size == 2*n_sym and np.issubdtype(item.dtype, np.integer):
                  bits = item.astype(np.int64)
    else:
      # fallback: common signature (sig1, _, _, bit_info)
      if sig1 is None or bits is None:
          try:
              sig1_try, _, _, bits_try = out
              if sig1 is None and np.iscomplexobj(sig1_try) and sig1_try.size == sig_len:
                  sig1 = sig1_try.astype(np.complex64)
              if bits is None and np.issubdtype(np.asarray(bits_try).dtype, np.integer) and len(bits_try) == 2*n_sym:
                  bits = np.asarray(bits_try, dtype=np.int64)
          except Exception:
              pass

    if sig1 is None or bits is None:
        raise RuntimeError("Could not parse (sig1, bits) from generate_qpsk_signal(). Print its output to debug.")

    return sig1, bits

def encode_window_to_binary_vector(x_flat, coder, early_cutoff=None):
  x_t = torch.from_numpy(np.asarray(x_flat, dtype=np.float32))

  # --- NEW LOGIC: Handle Binary/Rate Encoders (mdelta) ---
  if getattr(coder, 'encoder', '') == 'mdelta':
      spike_array = coder.encode(x_t) # Returns 0s and 1s

      if early_cutoff is not None:
          # Reshape back to (TimeSteps, Channels) to apply cutoff
          n_channels = coder.n
          L = len(spike_array) // n_channels
          spike_matrix = spike_array.view(L, n_channels).clone()

          # early_cutoff is in symbols, multiply by 2 for interleaved I/Q steps
          cutoff_idx = early_cutoff * 2
          if cutoff_idx < L:
              spike_matrix[cutoff_idx:, :] = 0.0 # Zero out late spikes

          return spike_matrix.flatten().float()

      return spike_array.float()

  # --- OLD LOGIC: Nominal Path ---
  x_t = torch.from_numpy(np.asarray(x_flat, dtype=np.float32))
  spike_times = coder.encode(x_t)  # (window_len*2, n_gaussians)

  if early_cutoff is None:
      active = torch.isfinite(spike_times)
  else:
      active = torch.isfinite(spike_times) & (spike_times <= early_cutoff)

  return active.float().reshape(-1)

def encode_dataset_binary(X, coder, early_cutoff=None):
    return torch.stack(
        [encode_window_to_binary_vector(x, coder, early_cutoff=early_cutoff) for x in X],
        dim=0
    )

# ================== Inference Model Helpers ======================
def make_model_input(x_binary, num_neurons, device):
    return x_binary.to(device).unsqueeze(0).repeat(num_neurons, 1)

def make_input_labels(label, num_neurons, numDistalInputs, device):
    onehot = F.one_hot(torch.tensor(label, device=device), num_classes=num_classes).float()
    return onehot.unsqueeze(1).repeat(1, numDistalInputs)

# ================== Stats/Evaluation Helpers ======================
def check_input_sparsity(X_enc, name="Dataset"):

    total_elements = X_enc.numel()
    num_spikes = torch.sum(X_enc == 1).item()
    sparsity = (num_spikes / total_elements) * 100
    avg_spikes_per_window = torch.mean(torch.sum(X_enc, dim=1)).item()

    print(f"\n--- {name} Sparsity Check ---")
    print(f"Total items: : {total_elements}")
    print(f"  Total Spikes: {num_spikes}")
    print(f"  Sparsity Index: {sparsity:.2f}% (Target: 1-10% is usually healthy)")
    print(f"  Avg Spikes per Window: {avg_spikes_per_window:.2f} out of {X_enc.shape[1]}")
    print("-" * 30)

def track_firing_activity(winners, num_samples, min_length):
    winner_counts = np.bincount(winners, minlength=min_length)
    fire_rates = (winner_counts / num_samples) * 100
    print(f"--- Output Activity ---")
    for i, rate in enumerate(fire_rates):
        print(f"  Neuron {i} (Class {i}): Fired on {rate:.1f}% of samples")

def print_weight_stats(model, wMax1, wMax2):
    w1 = model.layer1.weights.detach().cpu().numpy()
    w2 = model.layer2.weights.detach().cpu().numpy()

    w_min1, w_max1 = w1.min(), w1.max()
    w_mean1, w_std1 = w1.mean(), w1.std()

    w_min2, w_max2 = w2.min(), w2.max()
    w_mean2, w_std2 = w2.mean(), w2.std()

    # Calculate % of weights that have reached the boundaries
    at_zero1 = (np.sum(w1 <= 0.1) / w1.size) * 100
    at_max1  = (np.sum(w1 >= (wMax1 - 0.1)) / w1.size) * 100

    at_zero2 = (np.sum(w2 <= 0.1) / w2.size) * 100
    at_max2  = (np.sum(w2 >= (wMax2 - 0.1)) / w2.size) * 100

    print(f"\n\n--- Layer 1: Weight Stats {w1.shape} ---")
    print(f"  Range: [{w_min1:.2f}, {w_max1:.2f}] | Mean: {w_mean1:.2f} | Std: {w_std1:.2f}")
    print(f"  Around Zero-valued synpase count: {at_zero1:.1f}% are ~0 (Useless synapses)")
    print(f"  wMax-valued synpase count: {at_max1:.1f}% are ~wMax (Strong pattern detectors)")

    print(f"--- Layer 2: Weight Stats {w2.shape} ---")
    print(f"  Range: [{w_min2:.2f}, {w_max2:.2f}] | Mean: {w_mean2:.2f} | Std: {w_std2:.2f}")
    print(f"  Around Zero-valued synpase count: {at_zero2:.1f}% are ~0 (Useless synapses)")
    print(f"  wMax-valued synpase count: {at_max2:.1f}% are ~wMax (Strong pattern detectors)\n\n")

# ================== Sketchy (Not Reliable) Calibration Helpers ======================
def calibrate_l1_threshold(model, X_enc, device, target_winner_frac=0.25, n_calib=500, min_s=1, max_s=100, steps=5):
    """
    Sweep threshold values and find the one where ~target_winner_frac
    of neurons fire per forward pass (e.g. 0.25 = 1 out of 4 neurons).
    """
    calib_idxs = torch.randperm(X_enc.shape[0])[:n_calib]
    best_threshold = model.layer1.threshold
    best_delta = float('inf')

    print(f"--- L1 Threshold Calibration (target winner frac: {target_winner_frac:.0%}) ---")

    for thresh in range(min_s, max_s, steps):
        model.layer1.threshold = thresh
        winner_counts = []

        with torch.no_grad():
            for idx in calib_idxs:
                x = X_enc[idx].to(device)
                data_in = make_model_input(x, model.layer1.numNeurons, device)
                _, l1_out = model.layer1(data_in)

                wta = model.layer1.wtaOutput.reshape(model.layer1.numNeurons, -1)
                best_per_neuron = wta.min(dim=1).values
                n_fired = (best_per_neuron < model.layer1.wMax).sum().item()
                winner_counts.append(n_fired / model.layer1.numNeurons)

        avg_frac = np.mean(winner_counts)
        delta = abs(avg_frac - target_winner_frac)

        print(f"  threshold={thresh:3d} -> avg winner frac: {avg_frac:.3f}  (delta: {delta:.3f})")

        if delta < best_delta:
            best_delta = delta
            best_threshold = thresh

        # early stop: once we've passed the target and delta is growing, stop
        if avg_frac < target_winner_frac * 0.5 and thresh > 5:
            break

    model.layer1.threshold = best_threshold
    print(f"\n  => Best L1 threshold: {best_threshold}  (winner frac delta: {best_delta:.3f})")
    return best_threshold
def calibrate_l2_threshold(model, X_enc, device, target_active_frac=0.3, n_calib=500, min_s=1, max_s=100, steps=5):
    """
    Sweep L2 threshold and find the value where ~target_active_frac
    of L2 segments produce non-wMax output times, giving the WTA
    something meaningful to discriminate between.
    """
    calib_idxs = torch.randperm(X_enc.shape[0])[:n_calib]
    best_threshold = model.layer2.threshold
    best_delta = float('inf')

    print(f"--- L2 Threshold Calibration (target active frac: {target_active_frac:.0%}) ---")

    for thresh in range(min_s, max(max_s, model.layer2.wMax * 1), steps):
        model.layer2.threshold = thresh
        active_fracs = []

        with torch.no_grad():
            for idx in calib_idxs:
                x = X_enc[idx].to(device)
                data_in = make_model_input(x, model.layer1.numNeurons, device)
                _, l1_out, _, l2_out, hidden_spikes = model(data_in)

                if hidden_spikes.sum().item() == 0:
                    continue

                wta2 = model.layer2.wtaOutput  # (numHiddenNeurons * numHiddenDendrites, numHiddenSegments)
                active = (wta2 < model.layer2.wMax).float().mean().item()
                active_fracs.append(active)

        if len(active_fracs) == 0:
            continue

        avg_active = np.mean(active_fracs)
        delta = abs(avg_active - target_active_frac)

        print(f"  threshold={thresh:3d} -> avg active seg frac: {avg_active:.3f}  (delta: {delta:.3f})")

        if delta < best_delta:
            best_delta = delta
            best_threshold = thresh

        if avg_active < target_active_frac * 0.3 and thresh > 5:
            break

    model.layer2.threshold = best_threshold
    print(f"\n  => Best L2 threshold: {best_threshold}  (active frac delta: {best_delta:.3f})")
    return best_threshold

## NeuTNN Classes

In [ ]:
class activeDendriteLayer_RNL(nn.Module):
    """
    Active Dendrite Layer with RNL Response Function

    Implements a single NeuTNN layer with multiple neurons, dendrites, and segments.
    Segments are assumed to be distal, while the proximal input is simply used as enable during weight update.
    Capable of online continuous learning via STDP/R-STDP.

    Args: numDistalInputs - Number of distal inputs to a single neuron. Within a neuron, the inputs are shared across all segments.
          numSegments     - Number of segments per dendrite per neuron.
          numDendrites    - Number of dendrites per neuron.
          numNeurons      - Number of neurons in the layer.
          threshold       - Excitation threshold for neuron. An output spike is generated when neuron's body potential reaches
                            this threshold.
          wMax            - Maximum value for synaptic weights.
          wInit           - Type of initialization for synaptic weights. Supports 'zero', 'uniform', and 'normal' initializations.
          rewardEn        - Enable reinforcement for STDP (R-STDP).
          device          - Device type (e.g. cpu, cuda).

    Input: data           - Input data to the layer of shape [numNeurons, numDistalInputs].

    __call__ is the forward function whereas "update" performs STDP update in either unsupervised or supervised fashion.
    """
    def __init__(self,
                 numDistalInputs: int,
                 numSegments: int,
                 numDendrites: int,
                 numNeurons: int,
                 threshold: int,
                 wMax: int = 7,
                 wInit: str = "zero",
                 rewardEn: bool = True,
                 device="cpu"
                ):

        super(activeDendriteLayer_RNL, self).__init__()

        self.numDistalInputs = numDistalInputs
        self.numSegments = numSegments
        self.numDendrites = numDendrites
        self.numNeurons = numNeurons
        self.threshold = threshold
        self.wMax = wMax
        self.wInit = wInit
        self.rewardEn = rewardEn
        self.device = device

        self.totalSegments = self.numNeurons*self.numDendrites*self.numSegments # Flattens the hierarchical biological structure into a single 1D number.
                                                                                # This allows PyTorch to process every segment across the entire layer
                                                                                # simultaneously using fast matrix operations instead of slow loops.

        # Initialize weights: weight matrix of shape [totalSegments, numDistalInputs]
        if self.wInit       == "zero":
            self.weights   = nn.Parameter(torch.zeros(self.totalSegments,
                                                      self.numDistalInputs), requires_grad=False)

        elif self.wInit     == "uniform":
            self.weights   = nn.Parameter(torch.randint(low = 0, high = self.wMax + 1,
                                          size=(self.totalSegments,
                                                self.numDistalInputs)).type(torch.FloatTensor),
                                          requires_grad=False)

        elif self.wInit     == "normal":
            self.weights   = nn.Parameter(torch.round(( (self.wMax+1)/2 + \
                                                        torch.randn(self.totalSegments, self.numDistalInputs))\
                                                      .clamp_(0,self.wMax)), requires_grad=False)

        self.increasingNum = torch.arange(1, self.wMax+1).repeat(self.totalSegments,self.numDistalInputs,1).to(device)
        # Creates a utility tensor containing arrays of [1, 2, 3... wMax]. This is used during the forward pass to simulate time steps or "Rank Order" spikes.


    ########################################### Inference function ###########################################
    def __call__(self, data):
        """
            Data is of shape [self.numNeurons, self.numDistalInputs]
          __call__ is the forward function whereas "update" performs STDP update in either unsupervised or supervised fashion.

        """
        # Takes input data and duplicates it so that every single segment across all dendrites has a copy of the input to evaluate locally.
        inputData = data.unsqueeze(1).repeat(1,self.numDendrites*self.numSegments,1)\
                        .reshape(self.totalSegments,self.numDistalInputs) # (totalSegments, numDistalInputs)

        # Applying weights -> Element-wise multiplication. inputs are binary (0 or 1) --> masks out the weights where the input is 0.
        weightedInput = inputData * self.weights

        # Converts the scalar weights into a simulated temporal sequence (up to wMax time steps)
        rnlResponse = torch.minimum(weightedInput.unsqueeze(2).repeat(1,1,self.wMax), self.increasingNum)

        # Weighted input summation - RNL
        summedOutput = torch.sum(rnlResponse, dim=1) # Sums the temporal responses across all distal inputs for each segment ~ membrane potential for the segment

        zeroCondition = summedOutput < self.threshold # Identifies segments whose summed membrane potential failed to reach the threshold.
        summedOutput[zeroCondition] = 0               # Kills the signal (sets to 0) for sub-threshold segments
        thresholdCrossed = summedOutput != 0          # boolean mask of segments that did successfully spike

        # Uses a cumsum trick to isolate the exact first time step a segment crossed the threshold.
        detectOutputTime = thresholdCrossed & (torch.cumsum(thresholdCrossed, 1) == 1)

        rnlOutputTimes = torch.max(detectOutputTime, dim=1)[1] # Extracts the integer index (the time step) of that first spike

        zeroIndices = torch.sum(zeroCondition,dim=1) == self.wMax
        rnlOutputTimes[zeroIndices] = self.wMax # If a segment never crossed the threshold at all, its spike time is set to wMax

        # Winner Take All inhibition across segments
        reshapedRnlOutput = rnlOutputTimes.reshape(-1,self.numSegments) # Groups the segment spike times back into their respective dendrites

        # Prepares a blank output tensor filled with wMax (the default "no spike" state).
        self.wtaOutput = torch.ones(self.numNeurons*self.numDendrites, self.numSegments,
                                     dtype=reshapedRnlOutput.dtype, device=self.device) * self.wMax

        # Winner-Take-All (WTA) mechanism. For each dendrite, it looks at all its segments and finds the one that spiked first (minimum time).
        minval, indices = torch.min(reshapedRnlOutput, dim=1)

        # Places only the winning segment's spike time into the output tensor. All the "losing" segments remain silenced at wMax.
        self.wtaOutput.scatter_(1, indices.unsqueeze(1), minval.unsqueeze(1))

        # Creates a binary mask showing which segments were the winners (time < wMax) and reshapes it to match the input matrix dimensions
        outputData = (self.wtaOutput < self.wMax).reshape(-1).unsqueeze(1).repeat(1,self.numDistalInputs)

        return inputData, outputData


    ########################################### Update function ###########################################
    def update(self, inputData, inputLabels, outputData, weights, capture, search, backoff, wBase, rewardEn, frac=0.25, leak_val=0.01):
        if rewardEn:
            captureVal = (inputData == 1) * (outputData == 1) * (inputLabels == 1) * capture + (inputData == 0) * (outputData == 1) * (inputLabels == 0) * capture * frac
            backoffVal = (inputData == 0) * (outputData == 1) * (inputLabels == 1) * backoff + (inputData == 1) * (outputData == 1) * (inputLabels == 0) * backoff * frac
            searchCondition  = (inputData == 1) * (outputData == 0) * (inputLabels == 1)
        else:
            captureVal = (inputData == 1) * (outputData == 1) * capture
            backoffVal = (inputData == 0) * (outputData == 1) * backoff
            searchCondition  = (inputData == 1) * (outputData == 0)

        # ---------------------------------------------------------
        # NEW: Synaptic Leak / Forgetting
        # If the output didn't spike, we apply a small decay.
        # ---------------------------------------------------------
        leakCondition = (outputData == 0)

        # Apply Capture (+), Backoff (-), and Leak (-)
        weights += captureVal - backoffVal - (leakCondition * leak_val)

        # Search
        weights[searchCondition] = torch.maximum(weights[searchCondition],
                                                  torch.minimum(weights[searchCondition] + search,
                                                                torch.tensor([wBase], device=self.device)))

        return weights.clamp_(0, self.wMax)

In [ ]:
class dualDendriteLayer_RNL(nn.Module):

  def __init__(self,
                numDistalInputs: int, numHiddenDistalInputs: int,
                numSegments: int, numHiddenSegments: int,
                numDendrites: int, numHiddenDendrites: int,
                numNeurons: int, numHiddenNeurons: int,
                threshold: int, thresholdHidden: int,
                capture: float, captureHidden: float,
                search: float, searchHidden: float,
                backoff: float, backoffHidden: float,
                frac: float, fracHidden: float,
                wBase: float, wBaseHidden: float,
                wMax: int = 7, wMaxHidden: int = 7,
                wInit: str = "zero", k_winners: int = 1,
                leak: float = 0.01, leakHidden: float = 0.0,
                device="cpu"):

      super(dualDendriteLayer_RNL, self).__init__()

      print(f"initializing dualDendriteLayer_RNL...")

      self.numDistalInputs=numDistalInputs
      self.numSegments=numSegments
      self.numDendrites=numDendrites
      self.numNeurons=numNeurons
      self.threshold=threshold
      self.capture=capture
      self.search=search
      self.backoff=backoff
      self.frac=frac
      self.wBase=wBase
      self.wMax=wMax
      self.wInit=wInit
      self.k_winners = k_winners
      self.leak = leak
      self.device=device

      self.numHiddenDistalInputs = numHiddenDistalInputs
      self.numHiddenSegments = numHiddenSegments
      self.numHiddenDendrites  = numHiddenDendrites
      self.numHiddenNeurons = numHiddenNeurons
      self.thresholdHidden = thresholdHidden
      self.captureHidden = captureHidden
      self.searchHidden = searchHidden
      self.backoffHidden = backoffHidden
      self.fracHidden = fracHidden
      self.wBaseHidden = wBaseHidden
      self.wMaxHidden = wMaxHidden
      self.leakHidden = leakHidden

      self.layer1 = activeDendriteLayer_RNL(
          numDistalInputs=numDistalInputs,
          numSegments=numSegments,
          numDendrites=numDendrites,
          numNeurons=numNeurons,
          threshold=threshold,
          wMax=wMax,
          wInit=wInit,
          rewardEn=rewardEn,
          device=device
      )
      self.layer2 = activeDendriteLayer_RNL(
          numDistalInputs=numHiddenDistalInputs,
          numSegments=numHiddenSegments,
          numDendrites=numHiddenDendrites,
          numNeurons=numHiddenNeurons,
          threshold=thresholdHidden,
          wMax=wMaxHidden,
          wInit=wInit,
          rewardEn=h_rewardEn,
          device=device
      )

      # At the end of __init__, after self.layer1 and self.layer2 are built:
      assert self.layer2.numDistalInputs == self.layer1.numNeurons, (
          f"layer2.numDistalInputs ({self.layer2.numDistalInputs}) must equal "
          f"layer1.numNeurons ({self.layer1.numNeurons}), "
          f"since hidden_spikes from L1 are fed as input to L2."
      )

  def forward(self, x, loud=False):
      """
      x: encoded spikes
      """
      l1_in, l1_out = self.layer1(x)

      # Take the min WTA output per neuron (across dendrites/segments)
      wta = self.layer1.wtaOutput.reshape(self.layer1.numNeurons, -1)

      # K-Winner-Take-All
      best_per_neuron = wta.min(dim=1).values  # (numNeurons,)

      # Safety check: don't ask for more winners than we have neurons
      k = min(self.k_winners, self.layer1.numNeurons)

      # Get the indices of the 'k' fastest neurons (smallest time values)
      # largest=False means we want the minimum values
      top_times, top_indices = torch.topk(best_per_neuron, k=k, largest=False)

      hidden_spikes = torch.zeros(self.layer1.numNeurons, device=self.device)

      # Allow all k winners to spike, provided they actually crossed the threshold
      for time, idx in zip(top_times, top_indices):
          if time < self.layer1.wMax:
              hidden_spikes[idx] = 1.0

      if loud:
        active_neurons = (hidden_spikes > 0).sum().item()
        total_neurons = hidden_spikes.numel()
        print(f"[DEBUG L1] Sparsity -> {active_neurons}/{total_neurons} neurons fired ({(active_neurons/total_neurons)*100:.1f}%)")

      l2_x = hidden_spikes.unsqueeze(0).repeat(self.layer2.numNeurons, 1)
      # l2_x = (num_neurons, num_neurons*(some value = 1 in our config))

      l2_in, l2_out = self.layer2(l2_x)
      # l2_out = (num_neurons, num_neurons*(some value = 1 in our config))

      return l1_in, l1_out, l2_in, l2_out, hidden_spikes

  def update(self, l1_in, l1_out, l2_in, l2_out, hidden_spikes, y):
    winner_idx = torch.argmax(hidden_spikes).item()

    # Guard: if no neuron fired (all-zero hidden_spikes), skip L1 weight update entirely.
    # Updating on a silent pass would incorrectly drive search on all segments.
    if hidden_spikes.sum().item() == 0:
        pass  # or: return early if you also want to skip L2
    else:
        # k-WTA STDP
        l1_out_inhibited = l1_out.clone()
        for n in range(self.layer1.numNeurons):
            # If this neuron did NOT spike, inhibit its segments
            if hidden_spikes[n] == 0.0:
                seg_start = n * self.layer1.numDendrites * self.layer1.numSegments
                seg_end   = seg_start + self.layer1.numDendrites * self.layer1.numSegments
                l1_out_inhibited[seg_start:seg_end] = 0

        l1_labels = make_input_labels(y, self.layer1.numNeurons, self.layer1.numDistalInputs, self.device)
        l1_labels = l1_labels.repeat_interleave(self.layer1.numDendrites * self.layer1.numSegments, dim=0)

        self.layer1.weights.data = self.layer1.update(
            inputData=l1_in,
            inputLabels=l1_labels,
            outputData=l1_out_inhibited,
            weights=self.layer1.weights.data,
            capture=self.capture,
            search=self.search,
            backoff=self.backoff,
            wBase=self.wBase,
            rewardEn=rewardEn,
            frac=self.frac,
            leak_val=self.leak,
        )

    # L2 update — inhibit all non-winning neurons before STDP
    l2_wta = self.layer2.wtaOutput.reshape(self.layer2.numNeurons, -1)  # (numNeurons, numDendrites*numSegments)
    per_neuron_best = l2_wta.min(dim=1).values                           # (numNeurons,)
    l2_winner = torch.argmin(per_neuron_best).item()

    l2_out_inhibited = l2_out.clone()
    for n in range(self.layer2.numNeurons):
        if n != l2_winner:
            seg_start = n * self.layer2.numDendrites * self.layer2.numSegments
            seg_end   = seg_start + self.layer2.numDendrites * self.layer2.numSegments
            l2_out_inhibited[seg_start:seg_end] = 0

    l2_labels = make_input_labels(y, self.layer2.numNeurons, self.layer2.numDistalInputs, self.device)
    l2_labels = l2_labels.repeat_interleave(self.layer2.numDendrites * self.layer2.numSegments, dim=0)

    self.layer2.weights.data = self.layer2.update(
        inputData=l2_in,
        inputLabels=l2_labels,
        outputData=l2_out_inhibited,
        weights=self.layer2.weights.data,
        capture=self.captureHidden,
        search=self.searchHidden,
        backoff=self.backoffHidden,
        wBase=self.wBaseHidden,
        rewardEn=h_rewardEn,
        frac=self.fracHidden,
        leak_val=self.leakHidden
    )

## Signed Quantized Latency Encoder

In [ ]:
class SignedQuantizedLatencyEncoder:
    """
    Signed quantized latency encoder.

    For an input window x of length L:
      - output has length 2L
      - first L entries encode positive magnitudes
      - second L entries encode negative magnitudes (magnitude of negativity)

    Encoding rule:
      - stronger magnitude -> earlier spike
      - weaker nonzero magnitude -> later spike
      - zero magnitude -> no spike (Inf)

    Global scaling:
      - fit() computes dataset-level positive and negative scales
      - encode() uses those same scales for every window
    """

    def __init__(self, num_spike_times=16, clip_quantile=1.0, eps=1e-12):
        """
        num_spike_times: number of discrete spike times, e.g. 16 gives times 0..15
        clip_quantile: optional robustness clipping in (0,1], e.g. 0.995
                       if 1.0, uses max magnitude
        eps: numerical safety
        """
        self.encoder = 'sqle'
        self.num_spike_times = num_spike_times
        self.tmax = num_spike_times - 1
        self.clip_quantile = clip_quantile
        self.eps = eps

        self.pos_scale = None
        self.neg_scale = None
        self.is_fitted = False

    def fit(self, X):
        """
        Fit global scaling bounds from training windows.

        X: tensor or array of shape [N, L]
        """
        X = torch.as_tensor(X, dtype=torch.float32)

        pos_vals = X[X > 0]
        neg_vals = (-X[X < 0])  # magnitude of negativity

        if pos_vals.numel() == 0:
            self.pos_scale = torch.tensor(1.0, dtype=torch.float32)
        else:
            if self.clip_quantile < 1.0:
                self.pos_scale = torch.quantile(pos_vals, self.clip_quantile)
            else:
                self.pos_scale = pos_vals.max()

        if neg_vals.numel() == 0:
            self.neg_scale = torch.tensor(1.0, dtype=torch.float32)
        else:
            if self.clip_quantile < 1.0:
                self.neg_scale = torch.quantile(neg_vals, self.clip_quantile)
            else:
                self.neg_scale = neg_vals.max()

        self.pos_scale = torch.clamp(self.pos_scale, min=self.eps)
        self.neg_scale = torch.clamp(self.neg_scale, min=self.eps)
        self.is_fitted = True
        return self

    def _magnitudes_to_spike_times(self, mags, scale):
        """
        mags: nonnegative tensor
        scale: positive scalar

        Returns:
          spike times in {0,...,tmax} for mags > 0
          Inf for mags == 0
        """
        mags = torch.clamp(mags / scale, 0.0, 1.0)

        spike_times = torch.full_like(mags, float("inf"))

        nonzero = mags > 0
        # stronger magnitude -> earlier spike
        # mags=1 -> 0, mags~0 -> tmax
        spike_times[nonzero] = torch.floor(self.tmax * (1.0 - mags[nonzero]))

        return spike_times

    def encode(self, x):
        """
        Encode one window.

        x: tensor/array shape [L]

        returns: tensor shape [2L]
                 [positive_channel, negative_channel]
        """
        if not self.is_fitted:
            raise RuntimeError("Encoder must be fit() on training data before encode().")

        x = torch.as_tensor(x, dtype=torch.float32)

        pos_mags = torch.clamp(x, min=0.0)
        neg_mags = torch.clamp(-x, min=0.0)  # magnitude of negativity

        pos_times = self._magnitudes_to_spike_times(pos_mags, self.pos_scale)
        neg_times = self._magnitudes_to_spike_times(neg_mags, self.neg_scale)

        return torch.cat([pos_times, neg_times], dim=0)

    def encode_batch(self, X):
        """
        Encode many windows.

        X: tensor/array shape [N, L]

        returns: tensor shape [N, 2L]
        """
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.stack([self.encode(x) for x in X], dim=0)

## Multi-Threshold Delta Modulation Encoder

In [ ]:
class MultiDeltaEncoder:

    """
    A variant of the Delta Modulation that uses multiple thresholds as a measure for spikes. This allows for weighted
    delta values tha can boost accuracy due to better representation of data.

    Pseudocode Details found: (Evaluation of Encoding Schemes on Ubiquitous Sensor Signal for Spiking Neural Network)[https://arxiv.org/html/2407.09260v1]

    Attributes:
        data        (tensor): The already normalized data, in the shape: (train_sample_count, 2*window_len)
        thetas       (float): The threshold value used to determine spike generation.
        num_delta   (int): number of thresholds per pair of I/Q
    """

    def __init__(self, thetas: tuple[list[float],list[float]], num_delta: int):

        """ Initializes the encoder: store values

        Args:
          thetas (float, float):    determines when a spike should be generated
          num_delta   (int): number of thresholds per pair of I/Q
        """

        self.encoder = 'mdelta' # encoder type
        self.n = num_delta * 4   # number of total channels, we separate positive and negative deltas
        self.thetas = (sorted(thetas[0], reverse=True), sorted(thetas[1], reverse=True)) # (thresholds[num_delta], thresholds[num_delta])
        self.count = {self.thetas[0][0]: 0, self.thetas[0][1]:0, self.thetas[0][2]:0}
        print(self.thetas)

    def count_summary(self):
      print(f"count summary: {self.count}")

    def encode(self, tensor: torch.Tensor):
      signal      = tensor.detach().cpu().numpy()
      L           = len(signal)
      num_delta   = self.n // 4
      spike_train = np.zeros((L, self.n), dtype=np.float32)

      this_count = self.count.copy()
      for i in range(L):
          prev_val = signal[i - 2] if i >= 2 else signal[i]  # same-channel lag
          diff     = signal[i] - prev_val

          if i % 2 == 0:   # I sample
              pos_base, neg_base = 0, num_delta
              thetas_to_use = self.thetas[0]
          else:             # Q sample
              pos_base, neg_base = 2 * num_delta, 3 * num_delta
              thetas_to_use = self.thetas[1]

          done = False
          for level, theta in enumerate(thetas_to_use):
            if done:
              continue
            if diff > theta:
                self.count[theta] +=1
                spike_train[i, pos_base + level] = 1.0
                done=True
            elif diff < -theta:
                self.count[theta] +=1
                spike_train[i, neg_base + level] = 1.0
                done=True

      return torch.from_numpy(spike_train.flatten())

## Signal Processing

In [ ]:
#PREPROCESSING
ALL_INTERFERENCE_TYPES = ['CommSignal2'] #['EMISignal1', 'CommSignal2', 'CommSignal3']

def _load_interf_window(sig_type, rng):
    """Load a random sig_len window from a random frame of the given type."""
    n_frames = n_train_frame_types[sig_type]
    chosen   = int(rng.integers(0, n_frames))
    data, _  = load_dataset_sample(chosen, "train_frame", sig_type)
    start    = int(rng.integers(0, max(1, len(data) - sig_len)))
    return data[start:start + sig_len].astype(np.complex64)

def _mix_with_phase(sig1, sig2, sinr_db, rng, use_phase=True):
    P_soi    = np.mean(np.abs(sig1) ** 2)
    P_interf = np.mean(np.abs(sig2) ** 2)
    gain     = np.sqrt(P_soi / (P_interf * 10 ** (sinr_db / 10) + 1e-12))
    phi      = float(rng.uniform(0.0, 1.0)) if use_phase else 0.0
    coeff    = (gain * np.exp(1j * 2 * np.pi * phi)).astype(np.complex64)
    return (sig1 + sig2 * coeff).astype(np.complex64)

def apply_preprocessing(sig1, mode, rng):
    if mode == 'original':
        sig2    = _load_interf_window(interference_sig_type, rng)
        sinr_db = float(np.random.choice(sinr_db_list))
        P_soi   = np.mean(np.abs(sig1) ** 2)
        P_int   = np.mean(np.abs(sig2) ** 2)
        coeff   = np.sqrt(P_soi / (P_int * 10 ** (sinr_db / 10)))
        return [(sig1 + coeff * sig2).astype(np.complex64)]

    elif mode == 'baseline':
        sig2    = _load_interf_window(interference_sig_type, rng)
        sinr_db = float(rng.uniform(*icassp24_sinr_range_db))
        return [_mix_with_phase(sig1, sig2, sinr_db, rng, use_phase=use_random_phase)]

    elif mode == 'ku_tii':
        sig2    = _load_interf_window(interference_sig_type, rng)
        sinr_db = float(rng.uniform(*icassp24_sinr_range_db))
        results = [_mix_with_phase(sig1, sig2, sinr_db, rng, use_phase=use_random_phase)]
        if interference_sig_type == 'CommSignal2' and rng.random() < ku_tii_aug_ratio:
            sinr_high = float(rng.uniform(*ku_tii_high_sinr_range_db))
            results.append(_mix_with_phase(sig1, sig2, sinr_high, rng, use_phase=use_random_phase))
        return results

    elif mode == 'tub':
        rand_type = ALL_INTERFERENCE_TYPES[int(rng.integers(0, len(ALL_INTERFERENCE_TYPES)))]
        sig2      = _load_interf_window(rand_type, rng)
        sinr_db   = float(rng.uniform(*icassp24_sinr_range_db))
        return [_mix_with_phase(sig1, sig2, sinr_db, rng, use_phase=use_random_phase)]

    else:
        raise ValueError(f"Unknown preprocessing_mode '{mode}'. Choose: original, baseline, ku_tii, tub")

# Hyperparameters

In [ ]:
debug = True                              # enable this to see a lot of print statements
graphing = True                           # enable this to see lots of graphs
tuning = False                            # When u wanna tune parameters (this is not 100% reliable tho)
sq_padded = False                         # determines how signal input looks like b4 pre-process (we don't set this to True anymore)

synaptic_pruning = False                  # synaptic pruning feature!
synaptic_pruning_modulo = 4               # how many epochs do we run before pruning

layer_one_leak = False                    # enables leaky ramp in layer 1
layer_two_leak = False                    # enables leaky ramp in layer 2

# --------- MITRF SIGNAL CONFIGURATIONS --------------------------------------------------------------------------
sinr_db_list = [-6, -3, 0, 3]             # [-12, -9, -6, -3, 0, 3]  # low interference — best tested config for Delta encoder
interference_sig_type = "CommSignal2"     # "EMISignal1", "CommSignal2", "CommSignal3"
include_interference = False              # if False, train on pure SOI (no interference)
matched_filter_enable = False             # if True, convolve full signal with matched filter
use_subset = True                         # saves memory by getting subset of the data samples
max_samples = 12000                       # max # samples to pull

# --------- GENERAL CONFIGRATIONS (BEST NOT TO CHANGE) -----------------------------------------------------------
num_classes = 4         # always consistent

rrc_beta = 0.5
rrc_span_symbols = 8             # length = span*sps + 1 = 8*16+1 = 129 taps
seed = 1                         # reproducibility

sig_len = 40960
n_sum = 2560                            # 5120 bits / 2 bits per QPSK symbol
sps_orig = 16
tau0_orig = 8
decim = 1                               # integer decimation factor (1 = no decimation). Must divide 16 AND 8.

# --------- PARAMETER CONFIGURATION (FREE TO MIX & MATCH) --------------------------------------------------------
test_frac = 0.25                      # fraction of dataset for testing
epochs     = 15                       # how many epochs to use

rewardEn = False                     # Layer 1: Unsupervised
h_rewardEn = True                    # Layer 2: Supervised
wInit = 'uniform'                    # weight initialization type
num_signals = 400                    # how many synthetic SOI messages to generate

encoder = 'mdelta'                   # encoding type
window_len = 32                      # window length AFTER decimation (integer)
early_cutoff = window_len-2          # inhibits later-in-time signals in a spec window
k = 4                                # # k-winning neurons for Layer 1

# Layer 1
capture = 0.3                        # increments if True Positive
search = 0.2                         # increments if False Negative
backoff = 0.1                        # decrements if False/True Negative
frac =    0.25                       # some scalar value for STDP learning
leak       = 0.000                   # increase to reduce the influence false positive spikes impact weight potential
wBase      = 0                       # start weight value --> increasing this boosts initial learning rate
threshold  = 90                      # threshold for firing --> increasing decreases overall neurons that fire in layer
wMax       = 7                       # clamped weight value  --> increasing this reduces max potential saturation, increases resolution too
num_neurons = 16
numDendrites = 6
numSegments  = 4

# Layer 2
h_capture = 0.3
h_search  = 0.1
h_backoff = 0.2
h_frac = 0.25
h_leak = 0.0000
h_wBase = 0
h_wMax = 7
thresholdHidden = 20
h_num_neurons = 4
h_numDendrites = 8
h_numSegments = 4

# Leaky Ramp Model L1
if layer_one_leak:
  threshold = 89
  leak       = 0.001

# Leaky Ramp Model L2
if layer_two_leak:
  thresholdHidden = 19
  h_leak = 0.0001


# --------- Encoder Parameters ----------------------------------------------------------------------------------

# sqle
sqle_num_spike_times = wMax
sqle_clip_quantile = 0.995

# mdelta
thetas = ([0.025, 0.05, 0.10], [0.025, 0.05, 0.10])
num_delta = 3

# ── Preprocessing mode selector ─────────────────────────────────────────────
# 'original' : existing behavior (no changes)
# 'baseline' : continuous SINR [-33,3] dB + random phase  ← all top teams
# 'ku_tii'   : baseline + extra CommSignal2 samples at high SINR  ← 1st place
# 'tub'      : baseline + random interference type per sample  ← 3rd place

alfredo_preprocessing     = False
preprocessing_mode        = 'ku_tii'
icassp24_sinr_range_db    = (min(sinr_db_list), max(sinr_db_list))
ku_tii_high_sinr_range_db = (0.0, 3.0)
ku_tii_aug_ratio          = 0.7
_pp_rng                   = np.random.default_rng(seed)
use_random_phase          = False # False for delta, True for TBR (just changes if we should use random_phase)

############################################## DO NOT EDIT THIS PLZ
numDistalInputs = 0  # typically (data_len * n_gaussians)
###################################################################


# Setup Model

In [ ]:
cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# ========================= BUILD DATASET ========================================================
decim = int(decim)
sps = sps_orig // decim
tau0 = tau0_orig // decim
window_len = int(window_len)

# if matched filtering enabled, run this block
if matched_filter_enable:
    g = rrc_pulse(beta=rrc_beta, sps=sps_orig, span_symbols=rrc_span_symbols)  # taps for ORIGINAL Fs
    h_mf = g[::-1].conj().astype(np.complex64)  # textbook matched filter taps
    print("\b--- matched_filter_enable = True ---")
    print(f"sps_orig={sps_orig}, sps={sps}, tau0_orig={tau0_orig}, tau0={tau0}, decim={decim}")
    print(f"h_mf shape: {h_mf.shape}, center tap: {len(h_mf)//2}")
    print("[MF] Using RRC approx g:", g.shape, "beta=", rrc_beta, "||g||=", float(np.linalg.norm(g)))

print("\n--- BUILDING Dataset ---")
W_list = [] # stores list of symbols representing signals => Tensor(num_signals, 2560,2,L)
y_list = [] # stores list of ground truth symbols representing signals => Tensor(num_signals, 2560)
bits_list = [] # stores list of ground truth bits representing signals => Tensor(num_signals, 5120)

for i in range(num_signals):
    # --- 1) Generate clean SOI + bits ---
    sig1, bits = generate_soi_and_bits()  # sig1 (40960,) complex values, bits (5120,)

    if alfredo_preprocessing:
      # add interference by doing input = og_signal + coefficient * interference_signal
      if include_interference:
          if debug:
              print(f"\n--- Setting: include_interference = True, mode={preprocessing_mode} ---")
          mixtures = apply_preprocessing(sig1, mode=preprocessing_mode, rng=_pp_rng)
      else:
          mixtures = [sig1]

      # --- 3–6) Process each mixture ---
      for j, sig_mixture in enumerate(mixtures):

          if matched_filter_enable:
              if debug and j == 0:
                  print("\n--- Setting: matched_filter_enable = True ---")
              sig_proc = matched_filter_full(sig_mixture)
          else:
              sig_proc = sig_mixture.astype(np.complex64)

          if decim > 1:
              if debug and j == 0:
                  print(f"\n--- Setting: Cutting signal after timestep {decim} ---")
              sig_proc = sig_proc[::decim]

          X = np.vstack([sig_proc.real, sig_proc.imag]).astype(np.float32)
          W, centers = slice_symbol_windows_X(X, sps=SPS, tau0=TAU0, window_len=window_len, n_sym=n_sym)
          y4 = bits_to_symbol_classes(bits)

          W_list.append(W)
          y_list.append(y4)
          bits_list.append(bits)

          if debug and (i == 0 or (i % max(1, num_signals//5) == 0)) and j == 0:
              print(f"[ex {i}] sig_proc len={sig_proc.shape[0]} X={X.shape} W={W.shape} y4={y4.shape} interference={include_interference} mode={preprocessing_mode}")

      sinr_db = 0
      if debug == True and (i == 0 or (i % max(1, num_signals//5) == 0)):
          print(f"[ex {i}] sig_proc len={sig_proc.shape[0]} X={X.shape} W={W.shape} y4={y4.shape} interference={include_interference} sinr_db={sinr_db}")
    else:

      # add interference by doing input = og_signal + coefficient * interference_signal
      if include_interference:
        n_frames = n_train_frame_types[interference_sig_type]
        chosen_idx = np.random.randint(n_frames)
        data, meta = load_dataset_sample(chosen_idx, "train_frame", interference_sig_type)
        start_idx = np.random.randint(0, len(data) - sig_len)
        sig2 = data[start_idx:start_idx+sig_len].astype(np.complex64)
        sinr_db = float(np.random.choice(sinr_db_list))
        coeff = np.sqrt(np.mean(np.abs(sig1)**2) / (np.mean(np.abs(sig2)**2) * (10**(sinr_db/10))))
        sig_mixture = sig1 + coeff * sig2
      else:
        sig_mixture = sig1

      sig_proc = sig_mixture.astype(np.complex64)

      # --- 3) Optional matched filter on full-rate signal ---
      if matched_filter_enable:
        if debug and i == 0:
          print("\n--- Setting: matched_filter_enable = True ---")
        sig_proc = matched_filter_full(sig_mixture)

      X_clean = np.vstack([sig1.real, sig1.imag]).astype(np.float32)  # X_clean = (2, sig_len/decim) => divides up real and imaginary parts into respsective rows
      X = np.vstack([sig_proc.real, sig_proc.imag]).astype(np.float32)  # X = (2, sig_len/decim) => divides up real and imaginary parts into respsective rows
      # X = [ [I0, I1, I2, I3, I4, ... Isig_len/decim],
      #       [Q0, Q1, Q2, Q3, Q4, ... Qsig_len/decim] ]

      # 1): calculates where each symbol "lives" (tau0 [aka offset] + sps [stride] * index)
      # 2) For each symbol, grab snippet of length = window_len
      # 3): Resulting shape =====================================> W=(n_sym, 2, wlen), C=(n_sym, )
      W, centers = slice_symbol_windows_X(X, sps=sps, tau0=tau0, window_len=window_len, n_sym=n_sym)
      # Example: Assume window_len = 3, sps = 1, tau0 = 0, n_sym is total # symbols we need to classify (2-bit v#alues)
        # W = [ [[I0, I1, I2], [Q0, Q1, Q2] ],
        #       [[I1, I2, I3], [Q1, Q2, Q3] ], ...  ]
        # C = [self explanatory]

      # --- 4) Convert ground truths to symbols ---
      # Example: bits = [0, 1, 0, 0, 1, 1, 0, 0] => [01, 00, 11, 00] => [1, 0, 2, 0]
      y4 = bits_to_symbol_classes(bits)  # bits (# bits in total,) -> (n_sym,) ==> this means 2 bits per symbol in our config

      W_list.append(W) # input values => size = [num_signals, [n_sym, 2, wlen]]
      y_list.append(y4) # symbols => size = (num_signals, n_sym)
      bits_list.append(bits) # bits => size = (num_signals, n_sym)

# ========================= Standalone dataset builder cell (symbol-centered windows) ============
W_all = np.concatenate(W_list, axis=0).astype(np.float32)      # (num_signals, 2, window_len)
y_all = np.concatenate(y_list, axis=0).astype(np.int64)        # (num_signals, n_sym)
bits_all = np.stack(bits_list, axis=0).astype(np.int64)        # (num_signals, n_sym)

print("\n=== OUTPUTS ===")
print("W_all.shape:", W_all.shape, W_all.dtype)
print("y_all.shape:", y_all.shape, y_all.dtype)
print("bits_all.shape:", bits_all.shape, bits_all.dtype)
print("===============")

rng = np.random.default_rng(seed) # random seed
N_total = W_all.shape[0] # (num_signals, )
indices = np.arange(N_total) # [0, 1, ... (num_signals)-1]
print(f"\nN_total: {N_total}, Size of indices: {len(indices)}")

if use_subset and max_samples < N_total:
    print("\n--- Setting: use_subset = True ---")
    print(f"Clamping from {N_total} to {max_samples} indices")
    indices = rng.choice(indices, size=max_samples, replace=False)

# N = min(max_samples, N_total) = total samples
W_use = W_all[indices].astype(np.float32)   # (N, 2, window_len)
y_use = y_all[indices].astype(np.int64)     # (N,)

# Flatten: (N, 2, window_len) -> (N, window_len*2) -> this is done because datatype is complex number (i, j)
N, C, Lwin = W_use.shape
assert C == 2, f"Expected 2 channels, got {C}"
X_flat = np.transpose(W_use, (0, 2, 1)).reshape(N, 2 * Lwin).astype(np.float32) # (N, window_len*2)

# --- 5) Divide into Test and Train Set ---
X_train, X_test, y_train, y_test = train_test_split(
    X_flat,
    y_use,
    test_size=test_frac,
    random_state=seed,
    stratify=y_use)

if debug and graphing:
    unique_classes = np.unique(y_train)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, sym_class in enumerate(unique_classes):
        if i >= 4:
            break

        class_indices = np.where(y_train == sym_class)[0]
        ax = axes[i]

        num_examples_to_plot = min(10, len(class_indices))
        for j in range(num_examples_to_plot):
            sample_idx = class_indices[j]
            sample_iq = X_train[sample_idx].reshape(-1, 2)

            I_wave = sample_iq[:, 0]
            Q_wave = sample_iq[:, 1]

            ax.plot(I_wave, color='tab:blue', alpha=0.6, linewidth=1.5, label='I' if j==0 else "")
            ax.plot(Q_wave, color='tab:orange', linestyle='--', alpha=0.6, linewidth=1.5, label='Q' if j==0 else "")

        ax.set_title(f'Symbol Class: {sym_class}', fontsize=12, fontweight='bold')

        # Updated Bounds
        ax.axhline(5.0, color='red', linestyle=':', alpha=0.5)
        ax.axhline(-5.0, color='red', linestyle=':', alpha=0.5)
        ax.axhline(0.0, color='green', linestyle=':', alpha=0.5)

        ax.grid(True, alpha=0.2)
        if i == 0:
            ax.legend(loc='upper right')

    fig.suptitle(f'Pre-Scaled I/Q Waveforms Grouped by Symbol ({num_examples_to_plot} Overlaid Samples)', fontsize=16)
    fig.supxlabel('Time Step (within window)', fontsize=14)
    fig.supylabel('Pre-Scaled Amplitude', fontsize=14)

    plt.ylim(-5.5, 5.5) # Expanded Y-axis for padding
    plt.tight_layout()
    plt.show()

# print statistics if neccessary
if debug:
  X_train_t = torch.tensor(X_train) if not isinstance(X_train, torch.Tensor) else X_train

  # 1. Separate I and Q columns
  I_all = X_train_t[:, 0::2]
  Q_all = X_train_t[:, 1::2]

  # if we wanted to print out some metrics for comparison and for fun
  print("\n\n=== I/Q Dataset Statistics (Pre-Scaling) ===")
  print(f"I - Mean:   {I_all.mean().item():.4f}")
  print(f"I - Median: {I_all.median().item():.4f}")
  print(f"I - Max:    {I_all.max().item():.4f}")
  print(f"I - 90th %: {torch.quantile(I_all, 0.90).item():.4f}")
  print("-" * 40)
  print(f"Q - Mean:   {Q_all.mean().item():.4f}")
  print(f"Q - Median: {Q_all.median().item():.4f}")
  print(f"Q - Max:    {Q_all.max().item():.4f}")
  print(f"Q - 90th %: {torch.quantile(Q_all, 0.90).item():.4f}")
  print("=====================================================\n")

# Fit on train only
x_min, x_max, x_range = fit_minmax_scaler(X_train)

# Apply to both train and test
X_train = apply_minmax_scaler(X_train, x_min, x_range, multiplier=1.0, clip=True)
X_test  = apply_minmax_scaler(X_test,  x_min, x_range, multiplier=1.0, clip=True)

if debug:
  print("=== I/Q Dataset Statistics (Post-Scaling) ===")
  print("X_train min:", float(X_train.min()))
  print("X_train max:", float(X_train.max()))
  print("X_test  min:", float(X_test.min()))
  print("X_test  max:", float(X_test.max()))

if debug and graphing:
    unique_classes = np.unique(y_train)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)
    axes = axes.flatten()

    for i, sym_class in enumerate(unique_classes):
        if i >= 4:
            break

        class_indices = np.where(y_train == sym_class)[0]
        ax = axes[i]

        num_examples_to_plot = min(10, len(class_indices))
        for j in range(num_examples_to_plot):
            sample_idx = class_indices[j]
            sample_iq = X_train[sample_idx].reshape(-1, 2)

            I_wave = sample_iq[:, 0]
            Q_wave = sample_iq[:, 1]

            ax.plot(I_wave, color='tab:blue', alpha=0.6, linewidth=1.5, label='I' if j==0 else "")
            ax.plot(Q_wave, color='tab:orange', linestyle='--', alpha=0.6, linewidth=1.5, label='Q' if j==0 else "")

        ax.set_title(f'Symbol Class: {sym_class}', fontsize=12, fontweight='bold')

        # Updated Bounds
        ax.axhline(5.0, color='red', linestyle=':', alpha=0.5)
        ax.axhline(-5.0, color='red', linestyle=':', alpha=0.5)
        ax.axhline(0.0, color='green', linestyle=':', alpha=0.5)

        ax.grid(True, alpha=0.2)
        if i == 0:
            ax.legend(loc='upper right')

    fig.suptitle(f'Scaled I/Q Waveforms [-1, 1] Grouped by Symbol ({num_examples_to_plot} Overlaid Samples)', fontsize=16)
    fig.supxlabel('Time Step (within window)', fontsize=14)
    fig.supylabel('Scaled Amplitude', fontsize=14)

    plt.ylim(-5.5, 5.5) # Expanded Y-axis for padding
    plt.tight_layout()
    plt.show()

# --- 6) Initialize encoders by type ---
if encoder == 'sqle':
  print(f"SQLE hyperparameters: num_spike_times={sqle_num_spike_times}, clip_quantile={sqle_clip_quantile}")
  coder = SignedQuantizedLatencyEncoder(
      num_spike_times=sqle_num_spike_times,
      clip_quantile=sqle_clip_quantile,
  )
  coder.fit(X_train)
elif encoder == 'mdelta':
  print(f"Multi-Threshold Delta hyperparameters: num_delta{num_delta}, thetas={thetas}")
  coder = MultiDeltaEncoder(
      num_delta=num_delta,
      thetas=thetas
  )

# X_train_enc: given num_train_samples, 2*window_len*n_gaussians, we essentially take
# [
#     Sample 1: [[gauss0: I0_enc, Q0_enc, ... I_0_enc, Q0_enc] ... [gauss7: I0_enc, Q0_enc, ... I_0_enc, Q0_enc]],
#     Sample 2: [[gauss0: I0_enc, Q0_enc, ... I_0_enc, Q0_enc] ... [gauss7: I0_enc, Q0_enc, ... I_0_enc, Q0_enc]],
#     ... and so on until we get to Sample (num_train_samples-1)
# ]
X_train_enc = encode_dataset_binary(X_train, coder, early_cutoff=early_cutoff) # ([num_train_samples, 2*window_len*n_gaussians])
X_test_enc  = encode_dataset_binary(X_test,  coder, early_cutoff=early_cutoff) # ([num_test_samples, 2*window_len*n_gaussians])
numDistalInputs = X_train_enc.shape[1] # (2*window_len*n_gaussians)

if debug and graphing:
    unique_classes = np.unique(y_train)

    # Create a 4-row, 2-column grid. Share X-axis down the columns.
    fig, axes = plt.subplots(nrows=min(4, len(unique_classes)), ncols=2, figsize=(14, 12), sharex='col')

    for i, sym_class in enumerate(unique_classes):
        if i >= 4:
            break

        # Find the VERY FIRST sample in the dataset that matches this class
        class_indices = np.where(y_train == sym_class)[0]
        if len(class_indices) == 0:
            continue
        sample_idx = class_indices[0]

        # --- 1. Get the Analog Wave ---
        sample_iq = X_train[sample_idx].reshape(-1, 2)
        I_wave = sample_iq[:, 0]
        Q_wave = sample_iq[:, 1]

        # --- 2. Get the Spikes ---
        spike_sample = X_train_enc[sample_idx]
        if torch.is_tensor(spike_sample):
            spike_sample = spike_sample.cpu().numpy()

        if encoder in ['tae', 'sf']:
            # 1. Reshape the flat array back to (window_len*2, 2)
            spike_matrix = spike_sample.reshape(-1, 2)

            # 2. Convert np.inf to 0.0 so matplotlib can plot the "silence"
            spike_matrix[np.isinf(spike_matrix)] = 0.0

            # 3. Separate the alternating I and Q time steps
            I_spikes_raw = spike_matrix[0::2]  # Evens
            Q_spikes_raw = spike_matrix[1::2]  # Odds

            # 4. Combine UP (Col 0) and DOWN (Col 1) into +1 / -1
            I_spikes = I_spikes_raw[:, 0] - I_spikes_raw[:, 1]
            Q_spikes = Q_spikes_raw[:, 0] - Q_spikes_raw[:, 1]

            y_label_spikes = 'Spike (+1/-1)'
        elif encoder == 'sqle':
            # 1. SQLE splits the array in half: First half = Positive, Second half = Negative
            L = len(spike_sample) // 2
            pos_times = spike_sample[:L]
            neg_times = spike_sample[L:]

            # 2. Separate the alternating I and Q values for both halves
            pos_I_times = pos_times[0::2]
            pos_Q_times = pos_times[1::2]

            neg_I_times = neg_times[0::2]
            neg_Q_times = neg_times[1::2]

            # 3. Helper to convert latency (0-15) into graphable height (1.0 to ~0.06)
            def latency_to_height(times, tmax=15):
                h = np.zeros_like(times, dtype=float)
                valid = ~np.isinf(times)
                # t=0 -> 1.0 (Tall/Strong) | t=15 -> 0.0625 (Short/Weak)
                h[valid] = (tmax - times[valid] + 1) / (tmax + 1)
                return h

            # Assuming your default num_spike_times = 16 (so tmax = 15)
            tmax = 15

            # 4. Combine into final arrays: Positives go up, negatives go down
            I_spikes = latency_to_height(pos_I_times, tmax) - latency_to_height(neg_I_times, tmax)
            Q_spikes = latency_to_height(pos_Q_times, tmax) - latency_to_height(neg_Q_times, tmax)

            y_label_spikes = 'Latency (Early=Tall)'
        elif encoder == 'mdelta':
            # 1. Reshape the flat sample back to (Total Time Steps, Channels)
            # Total Time Steps = window_len * 2 (because I and Q are interleaved)
            # n_channels = 12 (for 3 thresholds)
            n_channels = coder.n
            num_delta = n_channels // 4
            spike_matrix = spike_sample.reshape(-1, n_channels)

            # 2. Split the interleaved I and Q time steps
            I_rows = spike_matrix[0::2]  # Even indices are In-Phase
            Q_rows = spike_matrix[1::2]  # Odd indices are Quadrature

            # 3. Create a weighting array so stronger thresholds look taller on the plot
            # Level 0 -> 0.33 height, Level 1 -> 0.66, Level 2 -> 1.0
            weights = np.linspace(1/num_delta, 1.0, num_delta)

            # 4. Extract Positive and Negative contributions
            # For I: Pos is channels 0:3, Neg is 3:6
            I_pos = np.dot(I_rows[:, 0:num_delta], weights)
            I_neg = np.dot(I_rows[:, num_delta:2*num_delta], weights)

            # For Q: Pos is channels 6:9, Neg is 9:12
            Q_pos = np.dot(Q_rows[:, 2*num_delta:3*num_delta], weights)
            Q_neg = np.dot(Q_rows[:, 3*num_delta:4*num_delta], weights)

            # 5. Combine into final stem heights (+ for Up, - for Down)
            I_spikes = I_pos - I_neg
            Q_spikes = Q_pos - Q_neg

            y_label_spikes = 'Delta Intensity'
        else:
          try:
              n_feat = numDistalInputs // (2 * window_len)
              spike_iq = spike_sample.reshape(n_feat, window_len, 2)
              I_spikes = spike_iq.sum(axis=0)[:, 0]
              Q_spikes = spike_iq.sum(axis=0)[:, 1]
              y_label_spikes = 'Spike Count'
          except Exception as e:
              I_spikes = spike_sample[0::2][:window_len]
              Q_spikes = spike_sample[1::2][:window_len]
              y_label_spikes = 'Spike (+1/-1)'

        # --- PLOT ANALOG (Left Column) ---
        ax_analog = axes[i, 0]
        ax_analog.plot(I_wave, label='I Wave', color='tab:blue', linewidth=2)
        ax_analog.plot(Q_wave, label='Q Wave', color='tab:orange', linestyle='--', linewidth=2)
        ax_analog.set_title(f'Class {sym_class} - Analog Signal', fontsize=12, fontweight='bold')
        ax_analog.axhline(5.0, color='red', linestyle=':', alpha=0.3)
        ax_analog.axhline(-5.0, color='red', linestyle=':', alpha=0.3)
        ax_analog.axhline(0.0, color='green', linestyle=':', alpha=0.5)
        ax_analog.set_ylim(-5.5, 5.5)
        ax_analog.grid(True, alpha=0.3)
        ax_analog.set_ylabel('Amplitude [-5, 5]')
        if i == 0: ax_analog.legend(loc='upper right')

        # --- PLOT SPIKES (Right Column) ---
        ax_spikes = axes[i, 1]
        time_steps = np.arange(len(I_wave))

        # Offset I and Q slightly so the stems don't perfectly overlap
        ax_spikes.stem(time_steps - 0.1, I_spikes, linefmt='b-', markerfmt='bo', basefmt=" ", label='I Spikes')
        ax_spikes.stem(time_steps + 0.1, Q_spikes, linefmt='orange', markerfmt='o', basefmt=" ", label='Q Spikes')

        ax_spikes.set_title(f'Class {sym_class} - {encoder.upper()} Spike Train', fontsize=12, fontweight='bold')
        ax_spikes.axhline(0.0, color='black', linewidth=1, alpha=0.5)
        ax_spikes.grid(True, alpha=0.3)
        ax_spikes.set_ylabel(y_label_spikes)
        if i == 0: ax_spikes.legend(loc='upper right')

    # X-labels on bottom row only to keep it clean
    axes[-1, 0].set_xlabel('Time Step (within window)', fontsize=12)
    axes[-1, 1].set_xlabel('Time Step (within window)', fontsize=12)

    fig.suptitle('SNN Encoding Transformation Across All 4 QPSK Symbol Classes', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

y_train_t = torch.from_numpy(y_train).long()
y_test_t  = torch.from_numpy(y_test).long()

print("\n\nnumDistalInputs:",numDistalInputs)
print("X_train_enc:", X_train_enc.shape)  # ([num_train_samples, numDistalInputs])
print(f"X_train_enc spike count: {(X_train_enc == 1).sum().item()}")
check_input_sparsity(X_train_enc, f"{encoder} Train Set")
if encoder == "mdelta":
  print(f"{coder.count_summary()}")

# Running Model

In [ ]:
# ========================= RUN DATASET ========================================================
l2_distal_inputs = num_neurons  # = hidden_spikes.shape[0]
rnl_model = dualDendriteLayer_RNL(
    numDistalInputs=numDistalInputs, numHiddenDistalInputs=l2_distal_inputs, # if u think abt it matrix-wise
    numSegments=numSegments, numHiddenSegments=h_numSegments,
    numDendrites=numDendrites, numHiddenDendrites=h_numDendrites,
    numNeurons=num_neurons, numHiddenNeurons=h_num_neurons,
    threshold=threshold, thresholdHidden=thresholdHidden,
    capture=capture, captureHidden=h_capture,
    search=search, searchHidden=h_search,
    backoff=backoff, backoffHidden=h_backoff,
    frac=frac, fracHidden=h_frac,
    wBase=wBase, wBaseHidden=h_wBase,
    wMax=wMax, wMaxHidden=h_wMax,
    wInit=wInit, k_winners=k,
    leak=leak, leakHidden=h_leak,
    device=device
).to(device)

# if you wish to use threshold calibrators, here is the palce to invoke the functions ###############################

print(f"Thresholds -> L1: {threshold}, L2: {thresholdHidden}")

for ep in range(epochs):

    start = time.time()

    # ----------------------------------------- TRAINING PHASE ------------------------------------------------------
    shuffle=True
    idxs = torch.arange(X_train_enc.shape[0])
    if shuffle:
        idxs = idxs[torch.randperm(len(idxs))]

    first_time = True
    correct = 0
    total = 0
    silent_count = 0

    for idx in idxs: # traverse through each sample
        x = X_train_enc[idx] # (numDistalInputs, ) = 2*window_len*n_gaussians
        y = int(y_train_t[idx].item()) # ground truth, single value in _pair_to_class[]

        data_in = make_model_input(x, num_neurons=num_neurons, device=device) # (num_neurons, numDistalInputs)
        # x = [x(0), x(1), ... x(numDistalInputs-1)]
        # assume num_neurons=4, numDistalInputs=2 => 4x2
        # data_in =
        # [ [x0, x1],
        #   [x0, x1],
        #   [x0, x1],
        #   [x0, x1] ]

        first_time = (idx % 2000 == 0)

        # Forward Pass through both layers
        l1inData, l1outData, l2inData, l2outData, hidden_spikes = rnl_model.forward(data_in, loud=first_time)

        if(first_time == True):
          print(f"\tL1 Spikes: {hidden_spikes}\n")
          print(f"\thidden_spikes: {hidden_spikes.tolist()}")

        # get WTA  -> predict_class_from_wta
        neuron_times = rnl_model.layer2.wtaOutput.reshape(-1)
        if(first_time):
          print(f"rnl_model.layer2.wtaOutput (transposed): {rnl_model.layer2.wtaOutput.reshape(1, -1)}")

        neuron_times = rnl_model.layer2.wtaOutput  # (numNeurons*numDendrites, numSegments)
        # Best spike time per dendrite (min across segments)
        best_per_dendrite = neuron_times.min(dim=1).values  # (numNeurons*numDendrites,)
        # Reshape to (numNeurons, numDendrites) and take best dendrite per neuron
        best_per_neuron = best_per_dendrite.reshape(
            rnl_model.layer2.numNeurons,
            rnl_model.layer2.numDendrites
        ).min(dim=1).values  # (numNeurons,)

        if torch.all(best_per_neuron == rnl_model.layer2.wMax):
            pred = 0  # silent
        else:
            pred = torch.argmin(best_per_neuron).item()  # guaranteed in [0, numNeurons-1]

        if(first_time):
          print(f"\nGuessed: {pred} vs Ground Truth: {y}")
        correct += int(pred == y)
        total += 1

        # STDP Update
        with torch.no_grad():
            rnl_model.update(
                l1_in=l1inData, # (num_neurons, numDistalInputs)
                l1_out=l1outData, # (num_neurons, num_neurons)
                l2_in=l2inData, # (num_neurons, num_neurons)
                l2_out=l2outData, # (num_neurons, num_neurons)
                hidden_spikes=hidden_spikes,
                y=y, # ground truth
                )
        first_time = False

    # Update accuracy tracker
    train_acc = correct / max(total, 1)
    silent_frac = 1.0 * silent_count / max(total, 1)

    train_acc_proxy = train_acc
    train_silent_frac = silent_frac

    # Assuming layer.weights is your weight tensor and wMax is 15
    w_max_count = (rnl_model.layer1.weights == rnl_model.layer1.wMax).sum().item()
    w_zero_count = (rnl_model.layer1.weights == 0).sum().item()
    total_w = rnl_model.layer1.weights.numel()

    print(f"[DEBUG STDP] L1 Weights -> {w_max_count/total_w*100:.1f}% Maxed | {w_zero_count/total_w*100:.1f}% Dead")

    w_max_count = (rnl_model.layer2.weights == rnl_model.layer2.wMax).sum().item()
    w_zero_count = (rnl_model.layer2.weights == 0).sum().item()
    total_w = rnl_model.layer2.weights.numel()

    print(f"[DEBUG STDP] L2 Weights -> {w_max_count/total_w*100:.1f}% Maxed | {w_zero_count/total_w*100:.1f}% Dead")

    # Add this inside your training loop, triggering at the end of every epoch
    if synaptic_pruning and ep != 0 and ep % synaptic_pruning_modulo == 0:
        with torch.no_grad():
            print(f"[Epoch {ep}] Applying Hard Pruning!")
            # --- Layer 1 Pruning ---
            # Create a mask of weights that are strong enough to survive (e.g., > 1.0)
            l1_mask = (rnl_model.layer1.weights.data > 0.3).float()

            # Multiply weights by the mask to physically zero out the weak ones
            rnl_model.layer1.weights.data *= l1_mask

            # --- Layer 2 Pruning ---
            l2_mask = (rnl_model.layer2.weights.data > 1.0).float()
            rnl_model.layer2.weights.data *= l2_mask

            print(f"[Epoch {ep}] Applied Hard Pruning! Severed some synapses")

    # ----------------------------------------- TESTING PHASE ------------------------------------------------------
    preds = []
    trues = []
    winners = []
    correct = 0
    silent_count = 0
    first_time = True

    for i in range(X_test_enc.shape[0]):

        x = X_test_enc[i] # (numDistalInputs, ) = 2*window_len*n_gaussians
        y = int(y_test_t[i].item()) # ground truth, single value in _pair_to_class[]

        data_in = make_model_input(x, num_neurons=num_neurons, device=device) # (num_neurons, numDistalInputs)
        # x = [x(0), x(1), ... x(numDistalInputs-1)]
        # assume num_neurons=4, numDistalInputs=2 => 4x2
        # data_in =
        # [ [x0, x1],
        #   [x0, x1],
        #   [x0, x1],
        #   [x0, x1] ]

        # Forward Pass through both layers
        l1inData, l1outData, l2inData, l2outData, hidden_spikes = rnl_model.forward(data_in,loud=first_time)

        # get WTA  -> predict_class_from_wta
        neuron_times = rnl_model.layer2.wtaOutput.reshape(-1)
        if(first_time):
          print(f"rnl_model.layer2.wtaOutput (transposed): {rnl_model.layer2.wtaOutput.reshape(1, -1)}")

        neuron_times = rnl_model.layer2.wtaOutput  # (numNeurons*numDendrites, numSegments)
        # Best spike time per dendrite (min across segments)
        best_per_dendrite = neuron_times.min(dim=1).values  # (numNeurons*numDendrites,)
        # Reshape to (numNeurons, numDendrites) and take best dendrite per neuron
        best_per_neuron = best_per_dendrite.reshape(
            rnl_model.layer2.numNeurons,
            rnl_model.layer2.numDendrites
        ).min(dim=1).values  # (numNeurons,)

        if torch.all(best_per_neuron == rnl_model.layer2.wMax):
            pred = 0  # silent
        else:
            pred = torch.argmin(best_per_neuron).item()  # guaranteed in [0, numNeurons-1]

        if(first_time):
          print(f"\nGuessed: {pred} vs Ground Truth: {y}")

        if pred == y:
          correct += 1

        # update trackers
        preds.append(pred)
        trues.append(y)
        winners.append(pred)
        first_time = False

    track_firing_activity(winners, len(trues), rnl_model.layer2.numNeurons)

    acc = accuracy_score(trues, preds)
    cm = confusion_matrix(trues, preds)
    silent_frac = silent_count / max(len(trues), 1)

    # ----- DONE
    dt = time.time() - start

    test_acc = acc
    test_cm = cm
    test_silent_frac = silent_frac
    test_preds = np.array(preds)
    test_trues = np.array(trues)

    print(f"\n\nepoch {ep+1}/{epochs}")
    print(f"  train accuracy proxy: {train_acc_proxy:.4f}")
    print(f"  train silent frac   : {train_silent_frac:.4f}")
    print(f"  test accuracy       : {test_acc:.4f}")
    print(f"  test silent frac    : {test_silent_frac:.4f}")
    print(f"  time                : {dt:.2f}s")
    print("  confusion matrix:")
    print(test_cm)

    print_weight_stats(rnl_model, wMax, h_wMax)